In [ ]:
import torch 
from torch import nn
import torch.nn.functional as F
from torch import optim
from protgen.data import ProteinDataset
from protgen.models.transformer import Transformer
from protgen.models.classifier_guided import Net
from torch.utils.data import DataLoader
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

In [ ]:
#Load the datasets
train_data = ProteinDataset(path_to_data="./data/train.csv")
val_data = ProteinDataset(path_to_data="./data/val.csv")
test_data = ProteinDataset(path_to_data="./data/test.csv")

#Create DataLoaders
train_loader = DataLoader(train_data, batch_size=512, shuffle=True)
val_loader = DataLoader(val_data, batch_size=512, shuffle=False)
test_loader = DataLoader(test_data, batch_size=512, shuffle=False)

In [ ]:
model = Transformer()
#Load a trained model!
#model.load_state_dict(torch.load("model.pt"))
x = torch.randint(0, 23, (32, 129))

capture = {}

def save_hidden(module, input, output):
    
    capture["hidden"] = output.detach()  # Detach the output to avoid tracking gradients

handle = model.feed_forward_network.register_forward_hook(save_hidden)

for x in train_loader:
    model.eval()  
    with torch.no_grad():
        _ = model(x)
        hidden = capture["hidden"]


#after capturing the hidden states for all batches. remove the handle
handle.remove()